# Week 4: The Recency-Frequency Trade-off — Complete CLV Engine
## Pareto/NBD vs BG/NBD, Gamma-Gamma Monetary Model & Holdout Validation

**Masterclass:** [Week 4 Deep-Dive](https://balaviswanath-analytics.github.io/analytics_masterclass/week4_recency_frequency.html)

**Dataset:** CDNOW (bundled with `lifetimes`)

**What You'll Build:**
1. BG/NBD vs Pareto/NBD comparison
2. Gamma-Gamma monetary value model
3. Complete CLV = E[Transactions] x E[Monetary] x Margin
4. Calibration/holdout validation framework
5. Customer-level CLV ranking

In [ ]:
!pip install lifetimes pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print("Libraries loaded.")

---
## Part 1: Calibration-Holdout Split

The gold standard for validating CLV models: fit on past data (calibration), predict future data (holdout), compare predictions vs. actuals.

In [ ]:
from lifetimes.datasets import load_transaction_data
from lifetimes.utils import calibration_and_holdout_data, summary_data_from_transaction_data

txns = load_transaction_data()
print(f"Transactions: {len(txns):,}")
print(f"Date range: {txns['date'].min()} to {txns['date'].max()}")

# Split: calibration = first 39 weeks, holdout = last 13 weeks
cal_holdout = calibration_and_holdout_data(
    txns, 'id', 'date',
    calibration_period_end='1997-09-30',
    observation_period_end='1997-12-31',
    monetary_value_col='spent'
)

print(f"\nCustomers: {len(cal_holdout):,}")
print(f"Calibration columns: frequency_cal, recency_cal, T_cal, monetary_value_cal")
print(f"Holdout columns: frequency_holdout, duration_holdout")
cal_holdout.head()

---
## Part 2: BG/NBD vs Pareto/NBD — Model Comparison

Both models solve the same problem (predict purchases for non-contractual customers) but with different assumptions about the "death" process:

| | BG/NBD | Pareto/NBD |
|---|---|---|
| Death timing | After each transaction | Continuously between transactions |
| Computation | Faster | Slower (numerical integration) |
| Accuracy | Slightly less | Slightly more (in theory) |
| Practical difference | Small for most datasets | Small for most datasets |

In [ ]:
from lifetimes import BetaGeoFitter, ParetoNBDFitter

# === BG/NBD ===
bgf = BetaGeoFitter(penalizer_coef=0.001)
bgf.fit(cal_holdout['frequency_cal'], cal_holdout['recency_cal'], cal_holdout['T_cal'])

# === Pareto/NBD ===
pnbd = ParetoNBDFitter(penalizer_coef=0.001)
pnbd.fit(cal_holdout['frequency_cal'], cal_holdout['recency_cal'], cal_holdout['T_cal'])

print("=== BG/NBD Parameters ===")
for k, v in bgf.params_.items(): print(f"  {k}: {v:.4f}")
print(f"\n=== Pareto/NBD Parameters ===")
for k, v in pnbd.params_.items(): print(f"  {k}: {v:.4f}")

In [ ]:
# Predict holdout purchases with both models
holdout_weeks = cal_holdout['duration_holdout'].iloc[0]  # same for all

cal_holdout['pred_bgf'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    holdout_weeks, cal_holdout['frequency_cal'],
    cal_holdout['recency_cal'], cal_holdout['T_cal'])

cal_holdout['pred_pnbd'] = pnbd.conditional_expected_number_of_purchases_up_to_time(
    holdout_weeks, cal_holdout['frequency_cal'],
    cal_holdout['recency_cal'], cal_holdout['T_cal'])

# Compare: MAE
mae_bgf = (cal_holdout['pred_bgf'] - cal_holdout['frequency_holdout']).abs().mean()
mae_pnbd = (cal_holdout['pred_pnbd'] - cal_holdout['frequency_holdout']).abs().mean()

print("=== Holdout Validation: MAE ===")
print(f"  BG/NBD:      {mae_bgf:.4f}")
print(f"  Pareto/NBD:  {mae_pnbd:.4f}")
print(f"  Winner: {'BG/NBD' if mae_bgf < mae_pnbd else 'Pareto/NBD'}")

In [ ]:
# Visual: predicted vs actual by frequency bin
from lifetimes.plotting import plot_calibration_purchases_vs_holdout_purchases

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_calibration_purchases_vs_holdout_purchases(bgf, cal_holdout, ax=axes[0], n=30)
axes[0].set_title('BG/NBD: Cal vs Holdout', fontsize=13, fontweight='bold')

plot_calibration_purchases_vs_holdout_purchases(pnbd, cal_holdout, ax=axes[1], n=30)
axes[1].set_title('Pareto/NBD: Cal vs Holdout', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 3: Gamma-Gamma Monetary Model

BG/NBD and Pareto/NBD predict *how many* purchases. Gamma-Gamma predicts *how much* per purchase.

**Key assumption:** purchase frequency and monetary value are independent. We test this before fitting.

In [ ]:
from lifetimes import GammaGammaFitter

# Check independence assumption
repeat_buyers = cal_holdout[cal_holdout['frequency_cal'] > 0]
corr = repeat_buyers['frequency_cal'].corr(repeat_buyers['monetary_value_cal'])
print(f"Correlation between frequency and monetary value: {corr:.3f}")
print(f"Independence assumption {'holds' if abs(corr) < 0.3 else 'VIOLATED — proceed with caution'}")

# Fit Gamma-Gamma
ggf = GammaGammaFitter(penalizer_coef=0.001)
ggf.fit(repeat_buyers['frequency_cal'], repeat_buyers['monetary_value_cal'])

print(f"\n=== Gamma-Gamma Parameters ===")
for k, v in ggf.params_.items(): print(f"  {k}: {v:.4f}")

In [ ]:
# Expected average profit per transaction
repeat_buyers['exp_avg_profit'] = ggf.conditional_expected_average_profit(
    repeat_buyers['frequency_cal'], repeat_buyers['monetary_value_cal'])

print("=== Expected Average Profit ===")
print(repeat_buyers['exp_avg_profit'].describe().round(2))

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(repeat_buyers['exp_avg_profit'], bins=50, color='#2d6a4f', edgecolor='white', alpha=0.85)
ax.set_title('Predicted Average Profit per Transaction', fontsize=14, fontweight='bold')
ax.set_xlabel('Expected Profit ($)')
ax.set_ylabel('Customers')
plt.tight_layout()
plt.show()

---
## Part 4: Complete CLV Calculation

**CLV = E[Transactions in next T periods] x E[Profit per transaction] x Margin**

With a discount rate to account for the time value of money.

In [ ]:
# Full CLV using the BG/NBD + Gamma-Gamma combination
# lifetimes provides a convenience method

DISCOUNT_RATE = 0.01  # monthly discount rate (approx 12% annual)
MONTHS = 12

# Customer-level CLV (only for repeat buyers)
repeat_buyers['clv_12m'] = ggf.customer_lifetime_value(
    bgf,
    repeat_buyers['frequency_cal'],
    repeat_buyers['recency_cal'],
    repeat_buyers['T_cal'],
    repeat_buyers['monetary_value_cal'],
    time=MONTHS,
    discount_rate=DISCOUNT_RATE
)

print("=== 12-Month CLV Distribution ===")
print(repeat_buyers['clv_12m'].describe().round(2))

print(f"\nTotal 12-month CLV (all repeat buyers): ${repeat_buyers['clv_12m'].sum():,.0f}")
print(f"Top 10% of customers hold: {repeat_buyers['clv_12m'].quantile(0.9) / repeat_buyers['clv_12m'].mean():.1f}x the average CLV")

In [ ]:
# CLV distribution and Pareto principle
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(repeat_buyers['clv_12m'].clip(upper=repeat_buyers['clv_12m'].quantile(0.99)),
             bins=50, color='#264653', edgecolor='white', alpha=0.85)
axes[0].set_title('12-Month CLV Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('CLV ($)')
axes[0].set_ylabel('Customers')

# Cumulative CLV share (Pareto curve)
sorted_clv = repeat_buyers['clv_12m'].sort_values(ascending=False).values
cum_pct = np.cumsum(sorted_clv) / sorted_clv.sum() * 100
cust_pct = np.arange(1, len(sorted_clv)+1) / len(sorted_clv) * 100

axes[1].plot(cust_pct, cum_pct, color='#c45d3e', linewidth=2)
axes[1].plot([0,100],[0,100], 'k--', alpha=0.3, label='Perfect equality')
axes[1].axhline(y=80, color='gray', linestyle=':', alpha=0.5)
axes[1].axvline(x=20, color='gray', linestyle=':', alpha=0.5)
axes[1].set_title('CLV Concentration (Pareto Curve)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('% of Customers (ranked by CLV)')
axes[1].set_ylabel('% of Total CLV')
axes[1].legend()

plt.tight_layout()
plt.show()

# Find exact Pareto ratio
idx20 = int(len(sorted_clv) * 0.20)
pct_from_top20 = sorted_clv[:idx20].sum() / sorted_clv.sum() * 100
print(f"Top 20% of customers generate {pct_from_top20:.0f}% of total CLV")

In [ ]:
# Top 20 most valuable customers
top20 = repeat_buyers.nlargest(20, 'clv_12m')[
    ['frequency_cal', 'recency_cal', 'T_cal', 'monetary_value_cal',
     'exp_avg_profit', 'clv_12m']]
top20.columns = ['Frequency', 'Recency', 'T', 'Avg Order', 'E[Profit]', 'CLV_12m']
print("=== Top 20 Most Valuable Customers ===")
print(top20.round(2).to_string())

---
## Part 5: The Complete CLV Engine Summary

| Component | Model | Predicts |
|-----------|-------|----------|
| Transaction frequency | BG/NBD or Pareto/NBD | How many purchases in next T |
| Monetary value | Gamma-Gamma | Expected profit per purchase |
| Combined | CLV formula | Total expected value per customer |
| Validation | Cal/holdout split | Are predictions accurate? |

---
## Exercises

### Exercise 1: 24-Month CLV
Recompute CLV with `time=24`. How does the ranking change? Do the top customers stay the same?

### Exercise 2: Segment by CLV Tier
Create CLV tiers (Top 10%, Next 20%, Middle 40%, Bottom 30%). Profile each tier: what's their average frequency, recency, and monetary value?

### Exercise 3: Sensitivity Analysis
Vary the discount rate from 0.5% to 3% monthly. How sensitive is the total CLV estimate? Plot total CLV vs discount rate.

```python
rates = [0.005, 0.01, 0.015, 0.02, 0.025, 0.03]
total_clvs = []
for r in rates:
    clv = ggf.customer_lifetime_value(bgf, ...)  # fill in
    total_clvs.append(clv.sum())
```

---
## Key Takeaways

1. **BG/NBD vs Pareto/NBD**: difference is usually small; BG/NBD is faster and often sufficient
2. **Gamma-Gamma** adds the monetary dimension — frequency alone doesn't capture value
3. **Cal/holdout validation** is non-negotiable before deploying CLV models
4. **CLV concentration** follows Pareto: top 20% of customers drive ~60-80% of value
5. **This is the foundation** for all retention investment decisions in Q1

**Congratulations!** Weeks 1-4 give you a complete CLV engine. Weeks 5-13 build advanced diagnostics and specialized tools on top of this foundation.

**Next:** [Week 5 — Competing Risks](https://balaviswanath-analytics.github.io/analytics_masterclass/week5_competing_risks.html)